# Train Baseline viBERT - ViGoEmotions trên Kaggle

Notebook này fine-tune một pretrained language model cho bài toán multi-label emotion classification trên ViGoEmotions.

Sau khi train xong, notebook sẽ lưu checkpoint vào `/kaggle/working/vigo_baseline_outputs/<run_name>/best_model`, có thể dùng checkpoint đó để chạy notebook evaluate theo code-switching subset.

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn

In [ ]:
import os
import re
import ast
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, hamming_loss

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
# Chọn model ở đây.
# Các model nên chạy trước: xlm-r, phobert, mbert, visobert.

MODEL_KEY = "vibert"

MODEL_ZOO = {
    "xlm-r": "FacebookAI/xlm-roberta-base",
    "mbert": "google-bert/bert-base-multilingual-cased",
    "phobert": "vinai/phobert-base-v2",
    "visobert": "uitnlp/visobert",
    "vibert": "FPTAI/vibert-base-cased",
    "cafebert": "uitnlp/CafeBERT",
}

MODEL_NAME = MODEL_ZOO[MODEL_KEY]

# Sửa path này theo Kaggle Dataset bạn add vào notebook.
# Ví dụ nếu Kaggle Dataset hiện trong /kaggle/input/vigoemotions thì để như dưới.
DATA_DIR = "/kaggle/input/YOUR_VIGOEMOTIONS_DATASET"

MAX_LENGTH = 128
THRESHOLD = 0.5
NUM_LABELS = 28
SEED = 42

# Kaggle T4/P100 thường ổn với batch 16. Nếu bị OOM, giảm TRAIN_BATCH_SIZE xuống 8.
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 2

RUN_NAME = f"{MODEL_KEY}-vigoemotions"
OUTPUT_DIR = f"/kaggle/working/vigo_baseline_outputs/{RUN_NAME}"
BEST_MODEL_DIR = f"{OUTPUT_DIR}/best_model"

LABEL_NAMES = [
    "amusement", "excitement", "joy", "love", "desire", "optimism", "caring",
    "pride", "admiration", "gratitude", "relief", "approval", "realization",
    "surprise", "curiosity", "confusion", "fear", "nervousness", "remorse",
    "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger",
    "annoyance", "disapproval", "neutral"
]

id2label = {i: label for i, label in enumerate(LABEL_NAMES)}
label2id = {label: i for i, label in enumerate(LABEL_NAMES)}

MODEL_NAME, OUTPUT_DIR

## Load ViGoEmotions từ Kaggle Dataset

Trên Kaggle, add dataset ViGoEmotions vào notebook, rồi sửa biến `DATA_DIR` ở cell config cho đúng folder trong `/kaggle/input/...`.

Notebook hỗ trợ:

- 3 file riêng có tên chứa `train`, `val`/`dev`/`validation`, `test`
- hoặc 1 file chung có cột `split` hoặc `set`
- định dạng `.csv`, `.xlsx`, `.xls`, `.parquet`, `.json`, `.jsonl`

In [ ]:
def normalize_split_name(split_name):
    if split_name in ["validation", "dev"]:
        return "val"
    return split_name

def read_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        xls = pd.ExcelFile(path)
        if {"train", "val", "test"}.issubset(set(xls.sheet_names)):
            frames = []
            for sheet in ["train", "val", "test"]:
                df = pd.read_excel(path, sheet_name=sheet)
                df["split"] = sheet
                frames.append(df)
            return pd.concat(frames, ignore_index=True)
        return pd.read_excel(path, sheet_name=xls.sheet_names[0])
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in [".json", ".jsonl"]:
        return pd.read_json(path, lines=(suffix == ".jsonl"))
    raise ValueError(f"Unsupported file type: {path}")

def find_data_files(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"DATA_DIR does not exist: {data_dir}\n"
            "Set DATA_DIR to your Kaggle dataset folder, e.g. /kaggle/input/vigoemotions"
        )
    exts = {".csv", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"}
    files = [p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts]
    if not files:
        raise FileNotFoundError(f"No supported data files found under: {data_dir}")
    return files

def standardize_columns(df):
    df = df.copy()
    lower_map = {c.lower(): c for c in df.columns}

    if "text" not in df.columns:
        for candidate in ["comment", "sentence", "content", "clean_text"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "text"})
                break

    if "labels" not in df.columns:
        for candidate in ["label", "emotion", "emotions", "target", "targets"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "labels"})
                break

    if "split" not in df.columns:
        for candidate in ["set", "data_split"]:
            if candidate in lower_map:
                df = df.rename(columns={lower_map[candidate]: "split"})
                break

    missing = [c for c in ["text", "labels"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Available columns: {list(df.columns)}")

    return df

def parse_labels(x, num_labels=NUM_LABELS):
    if isinstance(x, np.ndarray):
        x = x.tolist()

    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception:
            x = [int(i) for i in re.findall(r"\d+", x)]

    if isinstance(x, (list, tuple)):
        x = list(x)
        unique_values = set(np.unique(x)) if len(x) else set()
        if len(x) == num_labels and unique_values.issubset({0, 1, 0.0, 1.0}):
            return [float(v) for v in x]

        y = np.zeros(num_labels, dtype=np.float32)
        for idx in x:
            idx = int(idx)
            if 0 <= idx < num_labels:
                y[idx] = 1.0
        return y.tolist()

    y = np.zeros(num_labels, dtype=np.float32)
    try:
        y[int(x)] = 1.0
    except Exception:
        pass
    return y.tolist()

files = find_data_files(DATA_DIR)
print("Found files:")
for f in files:
    print("-", f)

named_split_frames = []
single_frames = []

for file_path in files:
    df = standardize_columns(read_table(file_path))
    name = file_path.stem.lower()

    if "split" in df.columns:
        single_frames.append(df)
    elif "train" in name:
        df["split"] = "train"
        named_split_frames.append(df)
    elif any(k in name for k in ["val", "dev", "validation"]):
        df["split"] = "val"
        named_split_frames.append(df)
    elif "test" in name:
        df["split"] = "test"
        named_split_frames.append(df)

if single_frames:
    data = pd.concat(single_frames, ignore_index=True)
elif named_split_frames:
    data = pd.concat(named_split_frames, ignore_index=True)
else:
    raise ValueError(
        "Cannot infer splits. Use files named train/val/test, "
        "or provide a single file with column split/set."
    )

data = standardize_columns(data)
data["split"] = data["split"].astype(str).str.lower().map(normalize_split_name)
data["labels"] = data["labels"].apply(parse_labels)
data = data[["text", "labels", "split"]].copy()

print(data.shape)
print(data["split"].value_counts())
data.head()

In [ ]:
train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df = data[data["split"].isin(["val", "validation", "dev"])].reset_index(drop=True)
test_df = data[data["split"] == "test"].reset_index(drop=True)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "labels"]]),
    "validation": Dataset.from_pandas(val_df[["text", "labels"]]),
    "test": Dataset.from_pandas(test_df[["text", "labels"]]),
})

dataset

## Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_ds = dataset.map(tokenize_batch, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_ds

## Compute `pos_weight`

Các notebook gốc dùng `BCEWithLogitsLoss(pos_weight=...)` để giảm ảnh hưởng mất cân bằng nhãn. Cell này tính lại `pos_weight` từ train set.

In [ ]:
train_label_matrix = np.array(train_df["labels"].tolist(), dtype=np.float32)
positive_counts = train_label_matrix.sum(axis=0)
negative_counts = len(train_label_matrix) - positive_counts
pos_weight = negative_counts / np.clip(positive_counts, 1, None)
pos_weight = np.clip(pos_weight, 1.0, 50.0)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float)

pd.DataFrame({
    "label": LABEL_NAMES,
    "positive_count": positive_counts.astype(int),
    "pos_weight": pos_weight,
}).sort_values("positive_count").head(10)

## Model + Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",
)

class WeightedMultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
        "subset_accuracy": accuracy_score(labels, preds),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    seed=SEED,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = WeightedMultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

trainer.train()

## Evaluate Test Set

In [ ]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")

print("Validation metrics:")
print(val_metrics)
print("\nTest metrics:")
print(test_metrics)

In [ ]:
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

with open(f"{BEST_MODEL_DIR}/label_names.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_NAMES, f, ensure_ascii=False, indent=2)

with open(f"{OUTPUT_DIR}/metrics.json", "w", encoding="utf-8") as f:
    json.dump({"validation": val_metrics, "test": test_metrics}, f, ensure_ascii=False, indent=2)

print("Saved best model to:", BEST_MODEL_DIR)
print("Use this path in subset-eval notebook:")
print(BEST_MODEL_DIR)

## Predict Test and Save Raw Outputs

File prediction này hữu ích nếu muốn debug thêm hoặc tự tính metric ngoài notebook.

In [ ]:
pred_out = trainer.predict(tokenized_ds["test"])
test_logits = pred_out.predictions
test_probs = sigmoid(test_logits)
test_preds = (test_probs >= THRESHOLD).astype(int)
test_labels = np.array(test_df["labels"].tolist(), dtype=int)

pred_df = test_df[["text"]].copy()
for i, label in enumerate(LABEL_NAMES):
    pred_df[f"true_{label}"] = test_labels[:, i]
    pred_df[f"prob_{label}"] = test_probs[:, i]
    pred_df[f"pred_{label}"] = test_preds[:, i]

pred_path = f"{OUTPUT_DIR}/test_predictions.csv"
pred_df.to_csv(pred_path, index=False)
print("Saved predictions to:", pred_path)